# UrbanRisk AI — M2: Chennai Flood / Waterlogging Prediction
**Milestone 2 of 9** — Flood-probability model for +6h / +12h ahead, built honestly around a real data
constraint: there is no continuous, hourly, ground-truth "did zone X flood at hour Y" dataset for Chennai.

### What real data we found and how we use it
| Source | What it is | Role here |
|---|---|---|
| **GCC Inundation Points with Depth** (via OpenCity.in, public domain, sourced from Greater Chennai Corporation) | Hundreds of real GPS-tagged points where GCC documented inundation, with measured depth in inches | **Static flood-vulnerability feature** per zone — real evidence of where Chennai floods, not a time series |
| **Open-Meteo Historical Weather Archive** | Hourly rainfall, temperature, humidity, wind | Dynamic features + the basis for our proxy target |
| **Open-Meteo Elevation API** | 90m-resolution DEM (Copernicus GLO-90) | Static terrain feature per zone |
| **OpenStreetMap Overpass API** | Water body / waterway geometry | Distance-to-water static feature per zone |
| **IMD rainfall classification** (Heavy: 64.5–115.5mm/24h, Very Heavy: 115.6–204.4mm/24h, Extremely Heavy: >204.4mm/24h — India Meteorological Department's own published thresholds) | Documented rainfall-intensity categories | Basis for our **proxy flood-risk target** |

### The target label — read this before trusting any metric below
There is no dataset that says "zone X flooded at hour Y" continuously through time. So the target used here,
`flood_risk_{6h,12h}`, is defined as: **1 if the rainfall accumulated in the next 6h/12h window crosses a
documented IMD heavy-rain threshold (scaled proportionally to the window length), 0 otherwise.**

This is a transparent, documented **decision-support heuristic** — exactly the same honesty standard your
original brief asked for in the risk engine itself. It is NOT a verified flood outcome. A Precision/Recall/F1
number on this target tells you "how well does the model predict IMD-defined heavy rain," not "how well does it
predict an actual flood." The real GCC inundation-point data is used separately, as a *feature* (static
vulnerability), not as the label — keeping the one real ground-truth signal we have from being diluted into a
guess.

**Before using any metric from this notebook in a report or interview, say explicitly that the flood label is a
rainfall-threshold proxy, not confirmed flooding.** That one sentence is the difference between honest work and
an inflated claim.


## 1. Setup

In [ ]:
!pip install -q xgboost shap scikit-learn pandas numpy requests matplotlib

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta

pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (12, 4)

END_DATE = (datetime.utcnow() - timedelta(days=5)).strftime("%Y-%m-%d")
START_DATE = (datetime.utcnow() - timedelta(days=5) - timedelta(days=730)).strftime("%Y-%m-%d")
print("Pulling weather data from", START_DATE, "to", END_DATE)

ZONES = [
    ("CHN_CENTRAL",   "Chennai Central",    13.0827,  80.2707),
    ("CHN_MANALI",    "Manali",             13.1631,  80.2586),
    ("CHN_KODUNGYR",  "Kodungaiyur",        13.1275,  80.2470),
    ("CHN_ROYAPRM",   "Royapuram",          13.1143,  80.2947),
    ("CHN_WSHRMNPT",  "Washermanpet",       13.1167,  80.2833),
    ("CHN_PERAMBUR",  "Perambur",           13.1102,  80.2323),
    ("CHN_KOYAMBDU",  "Koyambedu",          13.0694,  80.1948),
    ("CHN_ANNANGR",   "Anna Nagar",         13.0850,  80.2101),
    ("CHN_AMBATTUR",  "Ambattur",           13.1143,  80.1548),
    ("CHN_AVADI",     "Avadi",              13.1147,  80.0970),
    ("CHN_NUNGMBKM",  "Nungambakkam",       13.0569,  80.2425),
    ("CHN_TNAGAR",    "T Nagar",            13.0418,  80.2341),
    ("CHN_MYLAPORE",  "Mylapore",           13.0339,  80.2619),
    ("CHN_ADYAR",     "Adyar",              13.0067,  80.2570),
    ("CHN_GUINDY",    "Guindy",             13.0067,  80.2206),
    ("CHN_VELACHRY",  "Velachery",          12.9756,  80.2200),
    ("CHN_PERUNGDI",  "Perungudi",          12.9647,  80.2422),
    ("CHN_SHOLINGL",  "Sholinganallur/OMR", 12.9010,  80.2279),
    ("CHN_TAMBARAM",  "Tambaram",           12.9249,  80.1000),
    ("CHN_PALLAVRM",  "Pallavaram",         12.9675,  80.1491),
]
zones_df = pd.DataFrame(ZONES, columns=["zone_id", "zone_name", "latitude", "longitude"])
print(f"{len(zones_df)} zones (same grid as M1)")


## 2. Real flood evidence — GCC documented inundation points
Downloading the actual GCC "Inundation Points with Depth of Inundation" KML (public domain, via OpenCity.in).


In [ ]:
INUNDATION_KML_URL = "https://data.opencity.in/dataset/022dd080-e927-40d7-897d-adf3ee98ad69/resource/ceddf53f-03c0-4866-8ba8-5e84c8007a85/download/814ca028-4c84-4bd0-aa67-6dbaeb9b6ba5.kml"

def fetch_gcc_inundation_points(url):
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    ns = {"kml": "http://www.opengis.net/kml/2.2"}
    root = ET.fromstring(r.content)
    records = []
    for pm in root.iter("{http://www.opengis.net/kml/2.2}Placemark"):
        data = {}
        for sd in pm.iter("{http://www.opengis.net/kml/2.2}SimpleData"):
            data[sd.get("name")] = sd.text
        if "F_LATITUDE" in data and "F_LONGITUDE" in data:
            records.append({
                "latitude": float(data["F_LATITUDE"]),
                "longitude": float(data["F_LONGITUDE"]),
                "depth_inches": float(data.get("DEPTH", 0) or 0),
                "remarks": data.get("F_REMARKS", ""),
            })
    return pd.DataFrame(records)

inundation_df = fetch_gcc_inundation_points(INUNDATION_KML_URL)
print(f"Loaded {len(inundation_df)} real GCC-documented inundation points")
inundation_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 7))
ax.scatter(inundation_df["longitude"], inundation_df["latitude"],
           s=inundation_df["depth_inches"] * 2, alpha=0.4, c="firebrick", label="GCC inundation point (size = depth)")
ax.scatter(zones_df["longitude"], zones_df["latitude"], marker="x", c="black", s=50, label="Zone centers")
ax.set_title("Real GCC-documented inundation points vs zone grid")
ax.legend()
plt.tight_layout()
plt.show()


## 3. Static feature — historical inundation density & depth per zone

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

RADIUS_KM = 1.5  # documented points within this radius count toward a zone's vulnerability

inundation_features = []
for _, zone in zones_df.iterrows():
    dists = haversine_km(zone["latitude"], zone["longitude"],
                          inundation_df["latitude"].values, inundation_df["longitude"].values)
    nearby = inundation_df[dists <= RADIUS_KM]
    inundation_features.append({
        "zone_id": zone["zone_id"],
        "historical_inundation_count": len(nearby),
        "historical_max_depth_in": nearby["depth_inches"].max() if len(nearby) else 0.0,
        "historical_mean_depth_in": nearby["depth_inches"].mean() if len(nearby) else 0.0,
    })

inundation_zone_df = pd.DataFrame(inundation_features)
inundation_zone_df.sort_values("historical_inundation_count", ascending=False)


## 4. Static feature — elevation (Open-Meteo Elevation API, keyless)

In [ ]:
def fetch_elevation(lats, lons):
    url = "https://api.open-meteo.com/v1/elevation"
    params = {"latitude": ",".join(map(str, lats)), "longitude": ",".join(map(str, lons))}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()["elevation"]

zones_df["elevation_m"] = fetch_elevation(zones_df["latitude"].tolist(), zones_df["longitude"].tolist())
zones_df[["zone_name", "elevation_m"]].sort_values("elevation_m")


## 5. Static feature — distance to nearest water body (OpenStreetMap Overpass API)

In [ ]:
def fetch_nearest_water_distance_km(lat, lon, radius_m=5000):
    query = f"""
    [out:json][timeout:30];
    (
      node["natural"="water"](around:{radius_m},{lat},{lon});
      way["natural"="water"](around:{radius_m},{lat},{lon});
      way["waterway"~"river|canal|stream"](around:{radius_m},{lat},{lon});
    );
    out center;
    """
    try:
        r = requests.post("https://overpass-api.de/api/interpreter", data={"data": query}, timeout=40)
        r.raise_for_status()
        elements = r.json().get("elements", [])
    except Exception as e:
        print(f"  Overpass query failed ({e}), leaving as NaN")
        return np.nan

    if not elements:
        return np.nan  # no water body within radius_m — not necessarily zero risk, just not found nearby

    dists = []
    for el in elements:
        if el["type"] == "node":
            dists.append(haversine_km(lat, lon, el["lat"], el["lon"]))
        elif "center" in el:
            dists.append(haversine_km(lat, lon, el["center"]["lat"], el["center"]["lon"]))
    return min(dists) if dists else np.nan

water_distances = []
for _, zone in zones_df.iterrows():
    print(f"Querying water bodies near {zone['zone_name']}...")
    d = fetch_nearest_water_distance_km(zone["latitude"], zone["longitude"])
    water_distances.append(d)
    time.sleep(1.5)  # Overpass public instance — be polite

zones_df["water_body_distance_km"] = water_distances
zones_df[["zone_name", "water_body_distance_km"]]


**Caveat on `water_body_distance_km`**: OSM coverage of Chennai's water bodies and canals is good for major
features (Cooum, Adyar, Buckingham Canal, major lakes) but incomplete for smaller drainage channels — treat this
as a useful but imperfect signal, not a survey-grade distance. A `NaN` means no OSM-tagged water feature was
found within 5km, which itself can be informative (or just missing OSM data) — inspect rather than blindly
imputing zero.


## 6. Merge static features into the zone table

In [ ]:
zones_df = zones_df.merge(inundation_zone_df, on="zone_id", how="left")
zones_df[["zone_name", "elevation_m", "water_body_distance_km",
          "historical_inundation_count", "historical_max_depth_in"]]


## 7. Fetch weather history per zone (same as M1)

In [ ]:
def fetch_weather_history(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": start_date, "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,pressure_msl,precipitation,rain,wind_speed_10m",
        "timezone": "Asia/Kolkata",
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"])
    return df.rename(columns={
        "time": "timestamp", "temperature_2m": "temperature", "relative_humidity_2m": "humidity",
        "pressure_msl": "pressure", "precipitation": "rainfall_1h", "wind_speed_10m": "wind_speed",
    })

all_frames = []
for _, zone in zones_df.iterrows():
    print(f"Fetching weather for {zone['zone_id']}...")
    w = fetch_weather_history(zone["latitude"], zone["longitude"], START_DATE, END_DATE)
    time.sleep(0.5)
    for col in ["elevation_m", "water_body_distance_km", "historical_inundation_count",
                "historical_max_depth_in", "historical_mean_depth_in"]:
        w[col] = zone[col]
    w["zone_id"] = zone["zone_id"]
    w["zone_name"] = zone["zone_name"]
    all_frames.append(w)

master = pd.concat(all_frames, ignore_index=True)
master["water_body_distance_km"] = master["water_body_distance_km"].fillna(master["water_body_distance_km"].median())
print("Master shape:", master.shape)


## 8. Clean & feature engineer (grouped by zone, no leakage)

In [ ]:
weather_cols = ["temperature", "humidity", "pressure", "rainfall_1h", "rain", "wind_speed"]

def clean_zone_group(g, cols, max_gap=3):
    g = g.sort_values("timestamp")
    g[cols] = g[cols].interpolate(method="linear", limit=max_gap, limit_direction="both")
    return g

master = master.groupby("zone_id", group_keys=False).apply(lambda g: clean_zone_group(g, weather_cols))
master = master.sort_values(["zone_id", "timestamp"]).reset_index(drop=True)
print("Remaining NaNs in weather columns:", master[weather_cols].isna().sum().sum())


In [ ]:
def add_flood_features(g):
    g = g.sort_values("timestamp").reset_index(drop=True)

    g["hour"] = g["timestamp"].dt.hour
    g["day_of_week"] = g["timestamp"].dt.dayofweek
    g["month"] = g["timestamp"].dt.month
    g["is_weekend"] = (g["day_of_week"] >= 5).astype(int)

    # Past rainfall accumulation (predictive features — backward-looking, no leakage)
    for w in [1, 3, 6, 12, 24]:
        g[f"rainfall_past_{w}h"] = g["rainfall_1h"].shift(1).rolling(window=w, min_periods=1).sum()

    for col in ["temperature", "humidity", "pressure", "wind_speed"]:
        for lag in [1, 3, 6, 12]:
            g[f"{col}_lag{lag}h"] = g[col].shift(lag)

    # Future rainfall accumulation (used ONLY to build the proxy target, never as a feature)
    # Reverse -> rolling sum -> reverse gives a forward-looking window sum ending h hours ahead
    for h in [6, 12]:
        reversed_rain = g["rainfall_1h"][::-1]
        forward_sum = reversed_rain.rolling(window=h, min_periods=h).sum()[::-1]
        g[f"future_rainfall_{h}h"] = forward_sum.shift(-1)  # window starts at t+1

    return g

df = master.groupby("zone_id", group_keys=False).apply(add_flood_features)
print("Shape after feature engineering:", df.shape)


## 9. Build the proxy flood-risk target
IMD "Heavy Rain" = 64.5–115.5mm in 24h. Scaled proportionally: 6h threshold ≈ 64.5 × (6/24) ≈ 16.1mm,
12h threshold ≈ 64.5 × (12/24) ≈ 32.25mm. This linear scaling is a simplification — real rainfall intensity
isn't uniform across a day — but it's transparent, documented, and easy to defend in an interview, unlike a
black-box label.


In [ ]:
IMD_HEAVY_RAIN_24H_MM = 64.5

THRESHOLDS = {
    6: IMD_HEAVY_RAIN_24H_MM * (6 / 24),
    12: IMD_HEAVY_RAIN_24H_MM * (12 / 24),
}
print("Proxy flood-risk thresholds (mm):", THRESHOLDS)

for h in [6, 12]:
    df[f"flood_risk_{h}h"] = (df[f"future_rainfall_{h}h"] >= THRESHOLDS[h]).astype(int)

for h in [6, 12]:
    rate = df[f"flood_risk_{h}h"].mean()
    print(f"Positive rate for flood_risk_{h}h: {rate:.4f} ({int(df[f'flood_risk_{h}h'].sum())} / {len(df)} rows)")


## 10. Prepare model matrix & chronological split

In [ ]:
feature_cols = (
    [f"rainfall_past_{w}h" for w in [1, 3, 6, 12, 24]] +
    [f"{c}_lag{l}h" for c in ['temperature', 'humidity', 'pressure', 'wind_speed'] for l in [1, 3, 6, 12]] +
    ["hour", "day_of_week", "month", "is_weekend"] +
    ["elevation_m", "water_body_distance_km", "historical_inundation_count",
     "historical_max_depth_in", "historical_mean_depth_in"]
)
target_cols = ["flood_risk_6h", "flood_risk_12h"]

model_df = df.dropna(subset=feature_cols + target_cols).reset_index(drop=True)
model_df["zone_id"] = model_df["zone_id"].astype("category")
model_feature_cols = feature_cols + ["zone_id", "latitude", "longitude"]

# need lat/lon on model_df — merge back in
model_df = model_df.merge(zones_df[["zone_id", "latitude", "longitude"]], on="zone_id", how="left")

print(f"Rows usable for modeling: {len(model_df)}")

unique_ts = sorted(model_df["timestamp"].unique())
n = len(unique_ts)
train_cutoff = unique_ts[int(n * 0.70)]
val_cutoff = unique_ts[int(n * 0.85)]

train_df = model_df[model_df["timestamp"] < train_cutoff]
val_df = model_df[(model_df["timestamp"] >= train_cutoff) & (model_df["timestamp"] < val_cutoff)]
test_df = model_df[model_df["timestamp"] >= val_cutoff]

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
X_train, X_val, X_test = train_df[model_feature_cols], val_df[model_feature_cols], test_df[model_feature_cols]


## 11. Models — Logistic Regression, Random Forest, XGBoost (per horizon)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score)

def evaluate_clf(y_true, y_prob, threshold, label):
    y_pred = (y_prob >= threshold).astype(int)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_true, y_prob) if y_true.nunique() > 1 else float("nan")
    pr_auc = average_precision_score(y_true, y_prob) if y_true.nunique() > 1 else float("nan")
    print(f"[{label}] Precision={precision:.3f} Recall={recall:.3f} F1={f1:.3f} "
          f"ROC-AUC={roc_auc:.3f} PR-AUC={pr_auc:.3f}")
    return {"Precision": precision, "Recall": recall, "F1": f1, "ROC_AUC": roc_auc, "PR_AUC": pr_auc}

clf_results = {}
trained_models = {}

# Logistic regression needs numeric zone_id + scaling
lr_feature_cols = feature_cols + ["latitude", "longitude"]  # drop categorical zone_id for LR
scaler = StandardScaler().fit(X_train[lr_feature_cols])

for h in [6, 12]:
    y_train = train_df[f"flood_risk_{h}h"]
    y_val = val_df[f"flood_risk_{h}h"]
    y_test = test_df[f"flood_risk_{h}h"]

    # --- Logistic Regression ---
    lr = LogisticRegression(max_iter=1000, class_weight="balanced")
    lr.fit(scaler.transform(X_train[lr_feature_cols]), y_train)
    y_prob_lr = lr.predict_proba(scaler.transform(X_test[lr_feature_cols]))[:, 1]
    clf_results[f"logreg_{h}h"] = evaluate_clf(y_test, y_prob_lr, 0.5, f"LogReg +{h}h")

    # --- Random Forest ---
    rf = RandomForestClassifier(n_estimators=300, max_depth=10, class_weight="balanced",
                                 random_state=42, n_jobs=-1)
    rf.fit(pd.get_dummies(X_train, columns=["zone_id"]), y_train)
    X_test_rf = pd.get_dummies(X_test, columns=["zone_id"]).reindex(
        columns=pd.get_dummies(X_train, columns=["zone_id"]).columns, fill_value=0)
    y_prob_rf = rf.predict_proba(X_test_rf)[:, 1]
    clf_results[f"rf_{h}h"] = evaluate_clf(y_test, y_prob_rf, 0.5, f"RandomForest +{h}h")

    # --- XGBoost ---
    scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    xgb = XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=42,
        eval_metric="aucpr", enable_categorical=True,
        scale_pos_weight=scale_pos_weight, early_stopping_rounds=30,
    )
    xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    y_prob_xgb = xgb.predict_proba(X_test)[:, 1]
    clf_results[f"xgboost_{h}h"] = evaluate_clf(y_test, y_prob_xgb, 0.5, f"XGBoost +{h}h")

    trained_models[h] = {"logreg": lr, "rf": rf, "xgboost": xgb, "scaler": scaler}

pd.DataFrame(clf_results).T


## 12. Precision-Recall curves (more informative than ROC when positives are rare)

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, h in zip(axes, [6, 12]):
    y_test = test_df[f"flood_risk_{h}h"]
    for name, key in [("LogReg", "logreg"), ("RandomForest", "rf"), ("XGBoost", "xgboost")]:
        model = trained_models[h][key]
        if key == "logreg":
            y_prob = model.predict_proba(scaler.transform(X_test[lr_feature_cols]))[:, 1]
        elif key == "rf":
            X_test_rf = pd.get_dummies(X_test, columns=["zone_id"]).reindex(
                columns=pd.get_dummies(X_train, columns=["zone_id"]).columns, fill_value=0)
            y_prob = model.predict_proba(X_test_rf)[:, 1]
        else:
            y_prob = model.predict_proba(X_test)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        ax.plot(recall, precision, label=name)
    ax.set_title(f"Precision-Recall — flood_risk_{h}h")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend()
plt.tight_layout()
plt.show()


## 13. SHAP explanation — XGBoost, +6h horizon

In [ ]:
import shap

h = 6
X_test_shap = X_test.copy()
X_test_shap["zone_id"] = X_test_shap["zone_id"].cat.codes

explainer = shap.TreeExplainer(trained_models[h]["xgboost"])
shap_values = explainer(X_test_shap)

shap.summary_plot(shap_values, X_test_shap, max_display=15, show=True)


Pay attention to where `historical_inundation_count` and `historical_max_depth_in` (the **real GCC data**)
rank versus the rainfall-threshold-derived features. If they rank low, the model may just be relearning the
rainfall threshold that defined the label — worth investigating, not just reporting the number.


## 14. Per-zone risk snapshot (uses real historical evidence + live features)

In [ ]:
h = 6
snapshot = test_df.copy()
snapshot["predicted_flood_prob_6h"] = trained_models[h]["xgboost"].predict_proba(X_test)[:, 1]

zone_summary = snapshot.groupby("zone_name").agg(
    mean_predicted_prob=("predicted_flood_prob_6h", "mean"),
    historical_inundation_count=("historical_inundation_count", "first"),
    elevation_m=("elevation_m", "first"),
).sort_values("mean_predicted_prob", ascending=False)

zone_summary


## 15. Save models & metadata

In [ ]:
import joblib
import json
import os

os.makedirs("models", exist_ok=True)

for h in [6, 12]:
    joblib.dump(trained_models[h]["xgboost"], f"models/flood_xgboost_{h}h.joblib")
    joblib.dump(trained_models[h]["rf"], f"models/flood_rf_{h}h.joblib")
    joblib.dump(trained_models[h]["logreg"], f"models/flood_logreg_{h}h.joblib")

zones_df.to_csv("models/flood_zones_static_features.csv", index=False)
inundation_df.to_csv("models/gcc_inundation_points_raw.csv", index=False)

metadata = {
    "target_definition": (
        "PROXY LABEL, not verified flooding. flood_risk_{h}h = 1 if rainfall in the next h hours "
        "meets or exceeds IMD Heavy Rain (64.5-115.5mm/24h) scaled proportionally to h. "
        "Source: India Meteorological Department published rainfall classification."
    ),
    "static_feature_source": (
        "historical_inundation_count / historical_max_depth_in / historical_mean_depth_in are derived from "
        "REAL Greater Chennai Corporation inundation-point data (public domain, via OpenCity.in), "
        "counted/aggregated within 1.5km of each zone center. This is genuine field evidence, unlike the target."
    ),
    "elevation_source": "Open-Meteo Elevation API (Copernicus DEM GLO-90, 90m resolution)",
    "water_distance_source": "OpenStreetMap via Overpass API — incomplete for minor drainage channels",
    "feature_columns": model_feature_cols,
    "target_columns": target_cols,
    "test_metrics": clf_results,
    "train_range": [str(train_df['timestamp'].min()), str(train_df['timestamp'].max())],
    "test_range": [str(test_df['timestamp'].min()), str(test_df['timestamp'].max())],
}

with open("models/flood_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved M2 flood models + metadata to ./models/")
print(json.dumps(metadata["target_definition"], indent=2))
